# 강의 스크립트 EDA — 인터랙티브 노트북

NLP 과제 1 · AI 강의 분석 리포트 생성기 (1주차 데이터 탐색)

- 재사용 전처리 모듈(`src.preprocess`)을 그대로 불러 쓰고, 핵심 그래프를 인라인으로 확인한다.
- 전체 마크다운 리포트 + PNG는 맨 끝 셀에서 `build_report()`로 한 번에 생성한다.

> ⚠️ 원본 강의 텍스트 파생물 — 외부 공유 금지. 실행하려면 로컬에 제공 데이터가 있어야 한다.

In [ ]:
import sys, os
from pathlib import Path

# 프로젝트 루트를 import 경로에 추가
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['font.family'] = 'AppleGothic'  # macOS 한글 폰트
plt.rcParams['axes.unicode_minus'] = False

from src.preprocess.loader import build_dataset
from src.preprocess import text as textmod
from src.eda import report as R

## 1. 데이터 적재 & 개요

In [ ]:
df = build_dataset()
df['week'] = R.assign_week(df)
valid = df[~df['malformed']].copy()
print('총 발화:', len(valid), '| 강의일:', valid['date'].nunique(), '| 화자:', valid['speaker'].nunique())
print('총 글자 수:', int(valid['char_len'].sum()))
valid[['date','session','timestamp','speaker','subject','char_len','text']].head()

## 2. 발화량 — 일자·세션별

In [ ]:
piv = valid.pivot_table(index=valid['date'].dt.strftime('%m-%d'), columns='session',
                        values='text', aggfunc='count', fill_value=0).reindex(columns=['오전','오후'])
piv.plot(kind='bar', stacked=True, figsize=(10,4), color=['#4C72B0','#DD8452'])
plt.title('일자·세션별 발화 수'); plt.ylabel('발화 수'); plt.show()
display(piv.assign(합계=piv.sum(axis=1)))

## 3. 강의 시간 구조 (12시간제 → 24시간제 보정)
STT 타임스탬프는 AM/PM 표기가 없어 01~05시를 13~17시로 보정했다. 점심 공백이 드러난다.

In [ ]:
valid.groupby('hour').size().plot(kind='bar', figsize=(9,3.5), color='#C44E52')
plt.title('시간대(24h)별 발화 밀도'); plt.xlabel('시'); plt.ylabel('발화 수'); plt.show()

## 4. 발화 길이 분포 (발화 완결성 단서)

In [ ]:
valid['char_len'].clip(upper=300).plot(kind='hist', bins=50, figsize=(9,3.5), color='#8172B3')
plt.axvline(valid['char_len'].median(), color='k', ls='--')
plt.title('발화 길이 분포 (글자 수)'); plt.xlabel('글자 수'); plt.show()
valid['char_len'].describe()

## 5. 언어 표현 품질 — 필러 & 키워드 (KoNLPy)
최초 실행 시 JVM 기동으로 수 초~수십 초 소요된다.

In [ ]:
filler = textmod.filler_counts(valid['text'].tolist())
nouns = textmod.noun_counts(valid['text'].tolist())
print('필러 Top10:', filler.most_common(10))
print('키워드 Top15:', nouns.most_common(15))

## 6. ⚠️ 메타데이터–스크립트 내용 정합성 점검
메타데이터 과목명과 실제 발화 키워드를 비교한다. **둘은 일치하지 않는다** — 메타는 HTML/React/HTTP인데 실제 내용은 Java IO·SQL/DB다.

In [ ]:
rows = []
for d, g in valid.groupby('date'):
    kws = ', '.join(w for w,_ in textmod.noun_counts(g['text'].tolist()).most_common(6))
    rows.append((d.strftime('%m-%d'), g['subject'].iloc[0], str(g['content'].iloc[0])[:18], kws))
pd.DataFrame(rows, columns=['날짜','메타 과목','메타 내용','STT 실제 키워드']).set_index('날짜')

## 7. 전체 리포트 생성 (마크다운 + PNG)

In [ ]:
out = R.build_report()
print('생성 완료:', out)